In [ ]:
device = "3"
model_name = "meta-llama/Llama-3.2-3B"
dataset_size = 200
max_length = 8
evaluate_scores = True

In [3]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.append('../src')

In [ ]:
import torch
from utils import load_model_and_tokenizer, load_c4

experiment = "varying_lambda"

device = torch.device(f"cuda:{device}" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

model, tokenizer = load_model_and_tokenizer(model_name, device)

data = load_c4(tokenizer, dataset_size)
data[0]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Loading dataset from cache...


[1,
 1113,
 12215,
 2316,
 5507,
 1481,
 29493,
 1358,
 29510,
 1352,
 1115,
 3487,
 1032,
 1947,
 1392,
 1630,
 5185,
 6452,
 28095,
 1541,
 29491,
 781,
 1543,
 4052,
 2019,
 2842,
 3832,
 2619,
 1808,
 1150,
 1389,
 1134]

In [ ]:
from utils import (
    load_default_wepa_watermarker,
    run_experiment,
    get_wat_name,
    load_default_exp_watermarker,
)

degrees = [None, 1, 2]
lams = [16, 32, 64, 128, 256, 512, 1024, 2048, 4096]

for degree in degrees:
    for lam in lams:
        if degree is None:
            wat = load_default_exp_watermarker(model.config.vocab_size, device, lam=lam)
        else:
            wat = load_default_wepa_watermarker(
                model.config.vocab_size, device, degree=degree, lam=lam
            )
        wat_name = get_wat_name(wat)

        run_experiment(
            experiment=experiment,
            run_name=f"{model.name_or_path}_{wat_name}",
            wat=wat,
            model=model,
            tokenizer=tokenizer,
            data=data,
            max_length=max_length,
            device=device,
            evaluate_scores=evaluate_scores,
        )